# Portugal electricity market price profiles — 2025

**Delivery window:** 2025-03-19 through 2025-12-31; **bidding zone:** Portugal (PT). **Dependencies:** pandas, matplotlib. Run cells from top to bottom.

- **Day-Ahead:** genuine PT market prices from a public OMIE-derived historical archive; **secondary source**, not a verified direct OMIE API connection. Native hourly prices until 2025-09-30 and quarter-hourly from 2025-10-01. Archive records before October repeat hourly prices four times; those repeats are dropped.
- **IDA1, IDA2, IDA3:** each official OMIE auction file fetched independently; parsed only when expected date/period/price fields are present. Online fetching can be slow or unavailable. The OMIE parsing below is a *best-effort draft, not independently validated against a complete 2025 file*.
- **Continuous VWAP, aFRR/mFRR energy and capacity, imbalance:** plotted separately **only if you provide actual price CSVs**. No balancing-cost-to-consumer proxies, fake settlement prices, or invented values.

Sources: [historical PT archive](https://github.com/tiagofelicia/dados-energia) · [OMIE IDA archive](https://www.omie.es/en/file-access-list?dir=Intraday+auction+market+hourly+prices+in+Portugal&parents=%2FIntraday+Auction+Market%2F1.+Prices&realdir=marginalpibcpt).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from urllib.request import Request, urlopen
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import StringIO
import re

START = pd.Timestamp('2025-03-19', tz='Europe/Lisbon')
END = pd.Timestamp('2026-01-01', tz='Europe/Lisbon')  # exclusive
series = {}

def plot_price(label, data, unit='€/MWh'):
    if data.empty:
        print(label + ': no valid price observations'); return
    plt.figure(figsize=(13, 3))
    plt.plot(data.index, data.values, linewidth=.65)
    plt.title(f'{label} — Portugal | {len(data):,} observations')
    plt.ylabel(unit); plt.xlabel('Delivery time (Europe/Lisbon)')
    plt.grid(alpha=.25); plt.tight_layout(); plt.show()
    print(label, '|', 'min:', round(data.min(), 2), '| mean:', round(data.mean(), 2), '| max:', round(data.max(), 2))

def get_text(url):
    with urlopen(Request(url, headers={'User-Agent':'Mozilla/5.0'}), timeout=20) as r:
        return r.read().decode('utf-8-sig', errors='replace')

## 1. Day-Ahead — actual Portugal price series

The 2025 annual historical CSV is republished from OMIE-derived data. Use the UTC key to distinguish the repeated DST hour. Do not treat the four repeated quarter-hour rows before October as four independent market prices.

In [ ]:
URL = 'https://raw.githubusercontent.com/tiagofelicia/dados-energia/main/data/omie/historico/omie_historico_2025.csv'
try:
    raw = pd.read_csv(StringIO(get_text(URL)), low_memory=False)
    required = {'data_utc', 'preco_pt'}
    if not required.issubset(raw.columns):
        raise ValueError(f'Unexpected columns: {list(raw.columns)[:20]}')
    utc = pd.to_datetime(raw['data_utc'], utc=True, errors='coerce')
    price = pd.to_numeric(raw['preco_pt'], errors='coerce')
    da = pd.Series(price.values, index=utc).dropna()
    da = da.loc[da.index.notna()].sort_index()
    da = da.loc[(da.index >= START.tz_convert('UTC')) & (da.index < END.tz_convert('UTC'))]
    if da.empty: raise ValueError('No data in requested delivery period')
    # Before 2025-10-01, OMIE Day-Ahead was hourly; the source repeats each value 4 times.
    cutoff = pd.Timestamp('2025-10-01', tz='Europe/Lisbon').tz_convert('UTC')
    hourly = da.loc[da.index < cutoff].resample('h').first().dropna()
    quarter_hourly = da.loc[da.index >= cutoff]
    da = pd.concat([hourly, quarter_hourly]).sort_index()
    da.index = da.index.tz_convert('Europe/Lisbon')
    da.name = 'Day-Ahead PT'
    series['Day-Ahead PT'] = da
    plot_price('Day-Ahead PT', da)
except Exception as e:
    print('Day-Ahead unavailable — no values invented:', type(e).__name__, str(e)[:300])

## 2. Intraday Auctions — IDA1, IDA2, IDA3

Official OMIE archive: one file per auction session and date (prefix `marginalpibcpt`). File formats and delivery-period conventions must be checked against the official format specification before using the results for quantitative analysis. This downloader deliberately rejects unrecognized rows instead of guessing. To keep a quick test, first set `FETCH_ALL_IDA = False`; the default below downloads the entire date range.

In [ ]:
FETCH_ALL_IDA = True
PREFIX = 'marginalpibcpt'

def fetch_ida(day, auction):
    name = f'{PREFIX}_{day:%Y%m%d}{auction:02d}.1'
    url = f'https://www.omie.es/es/file-download?parents%5B0%5D={PREFIX}&filename={name}'
    try:
        body = get_text(url)
    except Exception:
        return []
    found = []
    for row in body.splitlines():
        p = [x.strip() for x in row.split(';')]
        if len(p) < 5 or not re.fullmatch(r'20\\d{2}', p[0]): continue
        try:
            delivery = pd.Timestamp(year=int(p[0]), month=int(p[1]), day=int(p[2]))
            period = int(p[3])
            val = float(p[4].replace(',', '.'))
            if not 1 <= period <= 100: continue
            # This archive's 'hourly prices' product is hourly; do not assume 15-min rows.
            t = delivery + pd.Timedelta(hours=period - 1)
            if START.date() <= t.date() < END.date():
                found.append((t, val))
        except (ValueError, TypeError, OverflowError):
            pass
    return found

days = pd.date_range('2025-03-18', '2025-12-31', freq='D') if FETCH_ALL_IDA else pd.DatetimeIndex([pd.Timestamp('2025-03-19')])
for auction in (1, 2, 3):
    jobs = [(day, auction) for day in days]
    records = []
    with ThreadPoolExecutor(max_workers=8) as pool:
        for res in pool.map(lambda x: fetch_ida(*x), jobs): records.extend(res)
    if not records:
        print(f'IDA{auction}: no parseable official observations; not plotted')
        continue
    idx = pd.DatetimeIndex([t for t, _ in records])
    vals = pd.Series([v for _, v in records], index=idx).sort_index()
    if vals.index.has_duplicates:
        print(f'IDA{auction}: repeated delivery periods found; review official file specification; not plotted')
        continue
    vals = vals.tz_localize('Europe/Lisbon', ambiguous='NaT', nonexistent='NaT').dropna()
    series[f'IDA{auction} PT (provisional parse)'] = vals
    plot_price(f'IDA{auction} PT — PROVISIONAL, not verified', vals)

## 3. All remaining distinct products — CSV inputs, no proxies

Put CSV files in a `data/` directory alongside this notebook, each with columns **`delivery_start`** (ISO timestamp **with a UTC offset**) and **`price_eur_mwh`**. For reserve capacity products, use **`price_eur_mw_h`** when that is the settlement unit; do not divide arbitrary consumer charges by activation volumes.

Missing products are reported, never silently represented by zeros.

In [ ]:
from pathlib import Path
PRODUCTS = {
    'Continuous Intraday VWAP': ('continuous_vwap_pt.csv', 'price_eur_mwh', '€/MWh'),
    'aFRR energy Up': ('afrr_energy_up_pt.csv', 'price_eur_mwh', '€/MWh'),
    'aFRR energy Down': ('afrr_energy_down_pt.csv', 'price_eur_mwh', '€/MWh'),
    'mFRR energy Up': ('mfrr_energy_up_pt.csv', 'price_eur_mwh', '€/MWh'),
    'mFRR energy Down': ('mfrr_energy_down_pt.csv', 'price_eur_mwh', '€/MWh'),
    'aFRR capacity Up': ('afrr_capacity_up_pt.csv', 'price_eur_mw_h', '€/MW/h'),
    'aFRR capacity Down': ('afrr_capacity_down_pt.csv', 'price_eur_mw_h', '€/MW/h'),
    'mFRR capacity': ('mfrr_capacity_pt.csv', 'price_eur_mw_h', '€/MW/h'),
    'Imbalance settlement': ('imbalance_pt.csv', 'price_eur_mwh', '€/MWh'),
}
for label, (filename, column, unit) in PRODUCTS.items():
    path = Path('data') / filename
    if not path.exists():
        print(label, ': missing price source:', path)
        continue
    try:
        f = pd.read_csv(path)
        if not {'delivery_start', column}.issubset(f): raise ValueError('Required columns absent')
        ts = pd.to_datetime(f['delivery_start'], utc=True, errors='raise').dt.tz_convert('Europe/Lisbon')
        v = pd.Series(pd.to_numeric(f[column], errors='coerce').values, index=pd.DatetimeIndex(ts)).dropna().sort_index()
        v = v[(v.index >= START) & (v.index < END)]
        if v.empty: raise ValueError('No valid prices in period')
        series[label] = v
        plot_price(label, v, unit)
    except Exception as e:
        print(label, ': invalid source, skipped:', e)

print('\\nProducts with plotted observations:', list(series))